In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

from sklearn.preprocessing import MultiLabelBinarizer
# used when each example can have multiple labels, as is true in this case 
# this converts a classifier into a vector. think action = 1, fantasy = 1, comedy = 0

from sklearn.feature_extraction.text import TfidfVectorizer
# turns text into numbers using TF-IDF
# vectorizer = TfidfVectorizer(stop_words = 'english', ngram_range = (1, 2))

from sklearn.multiclass import OneVsRestClassifier
# onevsrestclassifier is used for multilabel classification 
# rather than one giant model, onevsrest classifier trains one classifier for action vs not action, etc. 
# so, builds classifiers for each genre in this case. (we will probably have to fix this)

from sklearn.linear_model import LogisticRegression
# the actual learning algorithm. good thing i know logistic regression 

from sklearn.metrics import classification_report, f1_score
# for the evaluation metrics 

In [4]:
# import data
df = pd.read_csv("anime_dataset.csv")
# manga_df = pd.read_csv("manga_dataset.csv")
# df = pd.concat([anime_df, manga_df])

In [5]:
# cleaning steps 
# only really care about title_english, synopsis, genres 

df = df.dropna(subset = ['synopsis', 'genres'])
df = df[['title', 'title_english', 'synopsis', 'genres']] # need double brackets to subset a dataframe

# we use title as a fallback in case title_english doesn't exist 
df['title_english'] = df['title_english'].fillna(df['title'])
df = df[['title_english', 'synopsis', 'genres']] 
df

,title_english,synopsis,genres
0,Cowboy Bebop,"Crime is timeless. By the year 2071, humanity ...",Action|Award Winning|Sci-Fi
1,Cowboy Bebop: The Movie,"Another day, another bounty—such is the life o...",Action|Sci-Fi
2,Trigun,"Vash the Stampede is the man with a $$60,000,0...",Action|Adventure|Sci-Fi
3,Witch Hunter Robin,"Though hidden away from the general public, Wi...",Action|Drama|Mystery|Supernatural
4,Beet the Vandel Buster,It is the dark century and the people are suff...,Action|Adventure|Fantasy
...,...,...,...
30065,The Case Files of Biblia Bookstore,Hidden away in a corner of Kamakura is an unas...,Drama|Mystery
30066,The Case Files of Biblia Bookstore,Hidden away in a corner of Kamakura is an unas...,Drama|Mystery
30067,Around 40 Kenja no Isekai Seikatsu Nikki,Forty-year-old Satoshi Osako is a farmer in th...,Action|Adventure|Fantasy
30068,Galaxy Express 999: Claire of the Glass 3D,CG remake of the Ginga Tetsudou 999: Glass no ...,Drama|Sci-Fi


In [6]:
df['text'] = (df['title_english'] + " " + df['synopsis']) 

X_text = df['text'] # predictor variables

df["genre_list"] = df["genres"].apply(lambda x: x.split("|"))

In [7]:
df.to_csv("genre_list_included.csv") # so i can look at it. seems good 

In [8]:
# multi label binary coding is used when each example can have multiple labels, as is true in this case 
# this converts a classifier into a vector. think action = 1, fantasy = 1, comedy = 0
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['genre_list'])

In [11]:
# train/test split
df = df.reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(X_text, Y, test_size = 0.2, random_state = 1001)

# In an effort to avoid index issues
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=1)


In [12]:
# TF-IDF vectorization 
vectorizer = TfidfVectorizer(max_features = 5000, stop_words = "english", ngram_range=(1, 2))

# fit_transform ONLY on training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform ONLY on test data
X_test_tfidf = vectorizer.transform(X_test)

In [13]:
# training model 
model = OneVsRestClassifier(LogisticRegression(max_iter = 5000))
model.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...max_iter=5000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](21,)","[ 0, 1, 2,...,18,19,20]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...max_iter=5000), LogisticRegre...max_iter=5000), LogisticRegre...max_iter=5000), LogisticRegre...max_iter=5000), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,21
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,5000
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000


In [14]:
# making predictions 
y_pred = model.predict(X_test_tfidf)

In [15]:
# evaluate !!!!
micro_f1 = f1_score(y_test, y_pred, average="micro")
macro_f1 = f1_score(y_test, y_pred, average="macro")

####### evaluation metrics for multi-class classification 
# precision: how many predicted yes were actually yes. minimizing false positives 
# recall: measures the ability to find all actual positive cases, minimizing false negatives 
# harmonic mean: H = n / (sum from 1 to n of 1 / x_i)
# F1: a machine learning metric that acts as the harmonic mean of precision and recall 
# ranges from 0 to 1
# macro-f1 calculates F1-score for each class independently and averages them 
# micro-f1 aggregates total true positives, false negatives, false positives across all classes

print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}") 

Micro F1: 0.4466
Macro F1: 0.2354


In [16]:
# new_text = """A high school student discovers magical powers and must defend humanity from ancient demons."""

# new_text = """My Secret Depressed Husband A businessman decides to marry his secretary in order to
# keep good appearances, but they soon fall in love and grapple with 
# the awkwardness of revealing their true feelings. The secretary's dad dies.
# they live in a waste of a dystopia. the secretary has depression but the 
# businessman fights for her anyway"""

new_text = '''Super Stinky Trash Collectors Three friends in middle school decide to 
start collecting and recycling soda cans in order to raise money for their upcoming 
road trip, where they will make memories for a lifetime. '''

# Avant Garde|Award Winning|Drama|Sci-Fi|Suspense
# new_text = '''Fifteen years after a cataclysmic event known as the Second Impact, the world 
# faces a new threat: monstrous celestial beings called Angels invade Tokyo-3 one by one. 
# Mankind is unable to defend themselves against the Angels despite utilizing their most 
# advanced munitions and military tactics. The only hope for human salvation rests in the hands 
# of NERV, a mysterious organization led by the cold Gendou Ikari. NERV operates giant humanoid 
# robots dubbed "Evangelions" to combat the Angels with state-of-the-art advanced weaponry and 
# protective barriers known as Absolute Terror Fields. Years after being abandoned by his 
# father, Shinji Ikari, Gendou's 14-year-old son, returns to Tokyo-3. Shinji undergoes a 
# perpetual internal battle against the deeply buried trauma caused by the loss of his mother 
# and the emotional neglect he suffered at the hands of his father. Terrified to open himself 
# up to another, Shinji's life is forever changed upon meeting 29-year-old Misato Katsuragi, 
# a high-ranking '''

# new_text = '''An 18-year-old girl devotes herself to her religion and takes leave from school 
# in order to travel to South Korea for 18 months. She has many trials but ultimately learns
# the power of friendship and family, as well as the importance of God.'''

new_vector = vectorizer.transform([new_text])

# prediction = model.predict(new_vector) # default cutoff of 0.5
# predicted_genres = mlb.inverse_transform(prediction)
# print(predicted_genres)
# since this returned empty, that means model.predict(new_vector) returned all zeros 
# and the inverse transform returned empty, meaning no genres were predicted. 
# usually, the issue is that the default threshold is too strict. 

probabilities = model.predict_proba(new_vector)
for genre, prob in zip(mlb.classes_, probabilities[0]):
    if prob > 0.2:
        print(f"{genre}: {prob:.3f}")


Comedy: 0.587


In [17]:
threshold = 0.3 # for the longer synopsis honestly a 0.5, 0.6 might be fine 
prediction = (probabilities >= threshold).astype(int)
predicted_genres = mlb.inverse_transform(prediction)
print(predicted_genres)

[('Comedy',)]


In [18]:
# type(X_test) # this is a pandas series

# # I want a dataframe with the title of the anime, the text, the true genre, and the predicted genre 
# # where are the predicted genres stored? 

# predicted_genres_matrix = mlb.inverse_transform(y_pred)
# print(predicted_genres_matrix)

# # now i need these predicted genres in a dataframe...? won't worry about formatting correctly for now 

# predicted_genres_matrix[4]

# type(predicted_genres_matrix) # this is a list 

# predicted_df = 

print(df.index[:10])
print(X_test.index[:10])

RangeIndex(start=0, stop=10, step=1)
Index([2658, 10907, 1722, 1552, 8695, 3976, 2004, 19924, 21812, 8636], dtype='int64')


In [19]:
predicted_genres = mlb.inverse_transform(y_pred)
pred_df = pd.DataFrame({"predicted_genres": predicted_genres}, index = X_test.index)

print(pred_df.head())

actual_genres = mlb.inverse_transform(y_test)

results_df = pd.DataFrame({"actual_genres": actual_genres,"predicted_genres": predicted_genres}, index=X_test.index)

# print(results_df.head())

results_df = test_df[["title_english", "genres"]].copy()
results_df["predicted_genres"] = predicted_genres
results_df["predicted_genres"] = predicted_genres
results_df

          predicted_genres
2658                    ()
10907            (Comedy,)
1722   (Adventure, Sci-Fi)
1552     (Action, Fantasy)
8695                    ()


,title_english,genres,predicted_genres
17773,Rain Shower,Slice of Life,()
20072,Hyakushou Kizoku 4th Season,Comedy,"(Comedy,)"
6086,Ranma ½: 1994 Music Calendar,Action|Comedy,"(Adventure, Sci-Fi)"
6674,Paper Rabbit Rope x AU Collaboration,Comedy,"(Action, Fantasy)"
15555,Chained Soldier,Action|Fantasy|Ecchi,()
...,...,...,...
13183,Cardfight!! Vanguard Highschool Arc,Action,()
14942,The Irregular at Magic High School: Reminiscen...,Action|Fantasy|Sci-Fi,"(Fantasy, Supernatural)"
1900,Pops,Romance,"(Comedy,)"
11647,Shirome-chan no Uta,Comedy,()
